# Classical classifiers vs XGBoost — nested LOUO feature selection
**SVM (RBF / linear / poly), k-NN, and XGBoost** compared on the GroundWalks step data.

**Methodology (rigorous, leakage-free):**
- **Outer loop:** leave-one-**user**-out (LOUO). Each user is held out once as the test fold.
- **Inner loop:** within each outer fold's *training users*, **sequential forward selection (SFS)**
  picks features, scored by an **inner leave-one-user-out** CV (macro-F1). So feature selection
  never sees the outer test user — no leakage.
- **Per-classifier feature sets:** each classifier selects its own features, re-selected every
  outer fold (so there isn't one fixed set — we report selection *frequency* across folds).
- **Scaling:** StandardScaler fit on **train folds only** (SVM/k-NN need it; XGBoost doesn't but
  it's harmless). Applied inside every fold.
- **Class imbalance:** SVMs use `class_weight='balanced'`; report **macro-F1 + per-class F1**, not accuracy.
- **Classifiers are UNTUNED** (sensible defaults) — this is an out-of-the-box comparison, not each
  model's best-tuned performance. Hyperparameter tuning would be a further (nested) step.

**Honest caveats baked in:**
- Only 3–4 users → inner LOUO has 2–3 folds → SFS decisions are **noisy**; selected feature sets
  vary between outer folds. Differences between classifiers may be within fold-to-fold noise.
- SVM-RBF/poly are O(n²)–O(n³); the full run is heavy. `SUBSAMPLE_PER_CLASS` and `MAX_FEATURES`
  control cost. **Set `SUBSAMPLE_PER_CLASS=None` and `MAX_FEATURES=20` for the full run on your machine.**


In [13]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings('ignore')
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# ============================== CONFIG ==============================
DATA_DIR = '..'                 # folder with the *_dry.csv files
SCORING  = 'f1_macro'          # SFS inner-loop metric
MAX_FEATURES = 20              # SFS cap (full run); reduce to go faster
SFS_TOL = 0.002                # stop SFS when best add improves < this
SUBSAMPLE_PER_CLASS = 2500     # cap per-class training size (kernel SVM is O(n^2-3)); None = full (SLOW)
RANDOM_STATE = 0
SVM_MAX_ITER = 50000           # hard cap so a non-converging SVM fit STOPS instead of hanging
SFS_JOBS     = 4              # parallel feature-trials across cores (-1 = all)
# ====================================================================

# person-confound features to always exclude (match the model pipeline)
DROP = {'total_p_mean','total_pmax','impact_mag','impact_mag_norm','heel_impulse',
        'heel_mean_ratio','fore_mean_ratio','heel_pmax_ratio','fore_pmax_ratio'} \
       | {f'pressure_{i:02d}_mean_ratio' for i in range(1,13)}
META = {'foot','step_id','surface','source_user','source_recording'}

def load_task(task):
    files = {'2class':[('soft_dry.csv','soft'),('hard_dry.csv','hard')],
             '3class':[('grass_dry.csv','grass'),('loose_dry.csv','loose'),('paved_dry.csv','paved')]}[task]
    dfs=[]
    for fn,lab in files:
        d=pd.read_csv(f'{DATA_DIR}/{fn}'); d['surface']=lab; dfs.append(d)
    df=pd.concat(dfs,ignore_index=True)
    feats=[c for c in df.columns if c not in META and c not in DROP]
    return df, feats


In [14]:
# ---- classifier factory (UNTUNED defaults, clearly labeled) ----
def make_classifiers():
    return {
        'SVM-RBF'   : lambda: SVC(kernel='rbf',    C=1.0, gamma='scale', class_weight='balanced', max_iter=SVM_MAX_ITER, random_state=RANDOM_STATE),
        'SVM-linear': lambda: SVC(kernel='linear', C=1.0,                 class_weight='balanced', max_iter=SVM_MAX_ITER, random_state=RANDOM_STATE),
        'SVM-poly'  : lambda: SVC(kernel='poly',   C=1.0, degree=3, gamma='scale', class_weight='balanced', max_iter=SVM_MAX_ITER, random_state=RANDOM_STATE),
        'k-NN'      : lambda: KNeighborsClassifier(n_neighbors=7),
        'XGBoost'   : lambda: XGBClassifier(max_depth=4, n_estimators=250, learning_rate=0.05,
                                            subsample=0.9, colsample_bytree=0.8, eval_metric='mlogloss',
                                            random_state=RANDOM_STATE, verbosity=0),
    }

def macro_f1(yt, yp, labels): return f1_score(yt, yp, labels=labels, average='macro')

def fit_score(clf_factory, Xtr, ytr, Xte, yte, labels):
    # scale within fold (fit on train only)
    sc=StandardScaler().fit(Xtr)
    clf=clf_factory()
    # XGBoost needs integer labels
    if clf.__class__.__name__=='XGBClassifier':
        le=LabelEncoder().fit(labels); clf.fit(sc.transform(Xtr), le.transform(ytr))
        yp=le.inverse_transform(clf.predict(sc.transform(Xte)))
    else:
        clf.fit(sc.transform(Xtr), ytr); yp=clf.predict(sc.transform(Xte))
    return macro_f1(yte, yp, labels)


In [15]:
# ---- inner LOUO SFS: select features using ONLY the training users ----
def inner_louo_score(clf_factory, X, y, groups, cols, labels):
    # mean macro-F1 over inner leave-one-user-out folds, using feature columns `cols`
    scores=[]
    for u in np.unique(groups):
        tr=groups!=u; te=groups==u
        if len(np.unique(y[tr]))<len(labels): continue
        scores.append(fit_score(clf_factory, X[np.ix_(tr,cols)], y[tr], X[np.ix_(te,cols)], y[te], labels))
    return np.mean(scores) if scores else -1

from joblib import Parallel, delayed
def sfs_select(clf_factory, X, y, groups, labels, max_features=MAX_FEATURES, tol=SFS_TOL, n_jobs=SFS_JOBS):
    n=X.shape[1]; chosen=[]; remaining=list(range(n)); best=-1; history=[]
    while remaining and len(chosen)<max_features:
        scores=Parallel(n_jobs=n_jobs, prefer='processes')(
            delayed(inner_louo_score)(clf_factory, X, y, groups, chosen+[f], labels) for f in remaining)
        trial=sorted(zip(scores, remaining), reverse=True)
        bs,bf=trial[0]
        if bs-best < tol: break
        chosen.append(bf); best=bs; remaining.remove(bf); history.append((bf,round(bs,4)))
    return chosen, best, history


In [16]:
# ---- OUTER LOUO: per classifier, nested SFS then test on held-out user ----
def run_task(task):
    df, feats = load_task(task)
    print(f'=== {task}: {len(df)} steps, {len(feats)} candidate features, '
          f'users={sorted(df.source_user.unique())} ===')
    labels=sorted(df.surface.unique())
    users=sorted(df.source_user.unique())
    clsf=make_classifiers()
    results={name:{'fold_f1':[], 'sel_counts':{}} for name in clsf}
    for u in users:            # OUTER hold-out user
        tr_mask=(df.source_user!=u).values; te_mask=(df.source_user==u).values
        if len(np.unique(df.surface[tr_mask]))<len(labels): 
            print(f'  [skip outer user {u}: train missing a class]'); continue
        # optional subsample of TRAIN for SVM cost (test never subsampled)
        tr_df=df[tr_mask]
        if SUBSAMPLE_PER_CLASS:
            rng=np.random.default_rng(RANDOM_STATE); keep=[]
            for lab,sub in tr_df.groupby('surface'):
                ii=sub.index.values
                keep+=list(rng.choice(ii,min(len(ii),SUBSAMPLE_PER_CLASS),replace=False))
            tr_df=tr_df.loc[sorted(keep)]
        Xtr=tr_df[feats].fillna(0).values; ytr=tr_df.surface.values; gtr=tr_df.source_user.values
        Xte=df[te_mask][feats].fillna(0).values; yte=df[te_mask].surface.values
        for name,factory in clsf.items():
            t0=time.time()
            cols,inner_best,hist=sfs_select(factory,Xtr,ytr,gtr,labels)
            f1=fit_score(factory,Xtr[:,cols],ytr,Xte[:,cols],yte,labels)
            results[name]['fold_f1'].append(f1)
            for c in cols: 
                fn=feats[c]; results[name]['sel_counts'][fn]=results[name]['sel_counts'].get(fn,0)+1
            print(f'  outer={u:9} {name:11} nfeat={len(cols):2} inner={inner_best:.3f} '
                  f'test_macroF1={f1:.3f} ({time.time()-t0:.0f}s)')
    print(f'\n--- {task} summary (grouped LOUO macro-F1) ---')
    for name in clsf:
        arr=np.array(results[name]['fold_f1'])
        print(f'  {name:11} mean {arr.mean():.3f}  sd {arr.std():.3f}  folds {np.round(arr,3)}')
    return results


In [17]:
# ---- feature-selection frequency (which features are STABLE across folds) ----
def selection_report(results, task):
    print(f'\n=== {task}: feature selection frequency across outer folds ===')
    for name,r in results.items():
        n_folds=len(r['fold_f1'])
        if not n_folds: continue
        top=sorted(r['sel_counts'].items(), key=lambda kv:-kv[1])[:10]
        print(f'  {name}: ' + ', '.join(f'{f}({c}/{n_folds})' for f,c in top))


In [18]:
# ============ RUN ============
# For a quick check, keep MAX_FEATURES small / SUBSAMPLE set. For the real run on your
# RTX 5080 / 9700X box: set SUBSAMPLE_PER_CLASS=None and MAX_FEATURES=20 (already default).
res2 = run_task('2class'); selection_report(res2,'2class')
res3 = run_task('3class'); selection_report(res3,'3class')


=== 2class: 16270 steps, 49 candidate features, users=['Catherine', 'Kristian', 'Sam', 'Samantha'] ===
  outer=Catherine SVM-RBF     nfeat= 7 inner=0.794 test_macroF1=0.835 (69s)
  outer=Catherine SVM-linear  nfeat= 6 inner=0.785 test_macroF1=0.800 (48s)
  outer=Catherine SVM-poly    nfeat= 5 inner=0.779 test_macroF1=0.816 (50s)
  outer=Catherine k-NN        nfeat= 4 inner=0.751 test_macroF1=0.772 (3s)
  outer=Catherine XGBoost     nfeat= 4 inner=0.740 test_macroF1=0.794 (12s)
  outer=Kristian  SVM-RBF     nfeat= 8 inner=0.817 test_macroF1=0.604 (62s)
  outer=Kristian  SVM-linear  nfeat= 7 inner=0.823 test_macroF1=0.446 (47s)
  outer=Kristian  SVM-poly    nfeat= 3 inner=0.786 test_macroF1=0.716 (33s)
  outer=Kristian  k-NN        nfeat= 4 inner=0.779 test_macroF1=0.419 (3s)
  outer=Kristian  XGBoost     nfeat= 9 inner=0.809 test_macroF1=0.701 (23s)
  outer=Sam       SVM-RBF     nfeat= 5 inner=0.817 test_macroF1=0.715 (41s)
  outer=Sam       SVM-linear  nfeat= 6 inner=0.810 test_macroF1

In [19]:
# ---- final comparison table ----
import pandas as pd
def table(res, task):
    rows=[]
    for name,r in res.items():
        a=np.array(r['fold_f1'])
        if len(a): rows.append((task,name,round(a.mean(),3),round(a.std(),3),len(a)))
    return pd.DataFrame(rows,columns=['task','classifier','macroF1_mean','macroF1_sd','n_folds'])
cmp=pd.concat([table(res2,'2class'),table(res3,'3class')],ignore_index=True)
print(cmp.to_string(index=False))
cmp.to_csv('classifier_comparison.csv',index=False)
print('\nSaved classifier_comparison.csv')
print('\nREAD HONESTLY: with 3-4 users, macroF1_sd (fold spread) is often as large as the')
print('gaps between classifiers -> differences within ~1 sd are NOT meaningful. Classifiers are')
print('UNTUNED. 3-class loose is 3-user; treat rankings as indicative, not definitive.')


  task classifier  macroF1_mean  macroF1_sd  n_folds
2class    SVM-RBF         0.656       0.134        4
2class SVM-linear         0.582       0.161        4
2class   SVM-poly         0.726       0.054        4
2class       k-NN         0.595       0.145        4
2class    XGBoost         0.692       0.070        4
3class    SVM-RBF         0.353       0.066        4
3class SVM-linear         0.475       0.022        4
3class   SVM-poly         0.321       0.102        4
3class       k-NN         0.391       0.076        4
3class    XGBoost         0.476       0.067        4

Saved classifier_comparison.csv

READ HONESTLY: with 3-4 users, macroF1_sd (fold spread) is often as large as the
gaps between classifiers -> differences within ~1 sd are NOT meaningful. Classifiers are
UNTUNED. 3-class loose is 3-user; treat rankings as indicative, not definitive.
